### Dependencies

In [6]:
from apify_client import ApifyClient
from dotenv import load_dotenv
import pandas as pd
import re
import requests
import json
import os
import shutil
from datetime import datetime
from pathlib import Path
from apify_class import Apify

In [1]:
import os
def get_brand_folders(brands_path: str) -> list[str]:
    if not os.path.exists(brands_path):
        return []
    return [d for d in os.listdir(brands_path) if os.path.isdir(os.path.join(brands_path, d))]

brands_path = '../data/brands_LATER'
brands = get_brand_folders(brands_path)
print(brands)

['abdulsamadzia', 'chenoneofficial', 'dhillonenterprises365', 'dimensionsbyzakia', 'edition.pk', 'elo.shoponline', 'fashionoutlet.pk', 'halimay_florist', 'ittehad.official', 'leshahboutique', 'mahasphotographyofficial', 'mashclothing.pk', 'nomiansari', 'paperazzimagazine', 'preloved.clothes.khi', 'sairashakira', 'samsungpakistan', 'siasat.pk', 'tenadurrani', 'zahaonline']


In [7]:
import importlib
import apify_class

importlib.reload(apify_class)

<module 'apify_class' from 'c:\\Users\\ismai\\OneDrive\\Desktop\\UpClout\\src\\apify_class.py'>

## Save Profile Picture

In [ ]:
import requests

url = "https://instagram.fisb17-1.fna.fbcdn.net/v/t51.82787-19/525781976_18071951138078352_5065639200253104830_n.jpg?efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby4xMDgwLmMyIn0&_nc_ht=instagram.fisb17-1.fna.fbcdn.net&_nc_cat=100&_nc_oc=Q6cZ2gEVYdsLSF2tU9vYFPcHw_iXoZX9-klVLnAKky0N5CuNcq-nVZXbi2OhBIAtMSffUm8&_nc_ohc=nlBRVVj9-x8Q7kNvwE5CGHZ&_nc_gid=82PAI0sd7oaW1dRclrXGhg&edm=AP4sbd4BAAAA&ccb=7-5&oh=00_AfyLNdyV5AJgXBsYec5Tov01ZXgdONbQTpzisQoqrZfykQ&oe=69C87744&_nc_sid=7a9f4b"
# Send request
response = requests.get(url)
# Check if download was successful
if response.status_code == 200:
    with open("../data/_saras.archives/_saras.archives.jpg", "wb") as f:
        f.write(response.content)

## API Usage

In [ ]:
from dotenv import load_dotenv
import requests
import os, json

# Load environment variables
load_dotenv()

api_list = ['API_TOKEN_6','API_TOKEN_5', 'API_TOKEN_2', 'API_TOKEN_3', 'API_TOKEN', 'API_TOKEN_4'] 

for api in api_list:
    API_TOKEN = os.getenv(api)

    url = f"https://api.apify.com/v2/users/me?token={API_TOKEN}"
    res = requests.get(url)
    print(json.dumps(res.json(), indent=2))
"""
    data = res.json().get("data", {})

    used = data.get("monthlyUsageUsd", 0)
    limit = data.get("planMonthlyUsageLimitUsd", 5)  # fallback for free tier

    print(f"Used: ${used}")
    print(f"Limit: ${limit}")
    print(f"Remaining: ${limit - used}")"""

In [34]:
import psycopg2

def get_info(brand: str, influencers: list[str]):
    conn = psycopg2.connect(database="postgres", user="postgres", password=1040)
    cur = conn.cursor()

    query = """
        SELECT username, name, followers, following, location, businesscategoryname
        FROM influencers
        WHERE influencerid = %s
    """
    cur = conn.cursor()

    result = []

    for influencer in influencers:
        cur.execute(query, (influencer,))
        result.append(cur.fetchall())
    cur.close()
    
    return result

In [43]:
import json

with open('similarity_matches.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

influencers_for_brand = data.get('top_influencers_for_brand')
count = -1 
for brand, influencers in influencers_for_brand.items():
    count+=1
    if count <= 0:
        continue
    print(f'Brand: {brand}\nRec. Influencers: {influencers}')
    r = get_info(brand, influencers)
    for rr in r:
        print(rr)
    break

Brand: 1413557321
Rec. Influencers: ['68784811672', '54577601894', '308565584', '20561917045', '11625030684', '1764192723', '203003304', '67808433566', '72190000030', '45809188681']
[('toobanaveedd', 'Tooba', 10805, 146, 'islamabad', 'Personal blog')]
[('mahnoor.shahhh', 'NaN', 6973, 367, 'lahore', 'Digital creator')]
[('asimjofa', 'Asim Jofa', 2237050, 11, 'Nan', 'NaN')]
[('aenaakhan', 'Aena Khan', 1238321, 331, 'Nan', 'NaN')]
[('emannjafferr', 'Eman Jaffer', 26925, 411, 'karachi', 'Digital creator')]
[('hiramaniofficial', 'Hira Mani', 8520426, 589, 'Nan', 'NaN')]
[('naimalkhawarkhan', 'Naimal Khawar Abbasi', 3715918, 367, 'Nan', 'Artist')]
[('manahils.drafts', '𝐦𝐚𝐧𝐚𝐡𝐢𝐥.', 20873, 131, 'Nan', 'Digital creator')]
[('szmusa', 'sezen', 7621, 947, 'Nan', 'Artist')]
[('ayeshajahangirmalik', 'ᴀʏᴇꜱʜᴀ', 28683, 692, 'islamabad', 'None,Personal blog')]


In [9]:
import boto3
import requests
import os
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

BUCKET = os.getenv('AWS_S3_BUCKET')

def upload_profile_pic(username: str, image_url: str) -> str:
    """
    Downloads profile pic from Instagram URL,
    uploads to S3, returns the public URL.
    """
    # Download the image
    response = requests.get(image_url)
    if response.status_code != 200:
        return None

    # Upload to S3
    key = f"profile-pics/{username}.jpg"
    s3.put_object(
        Bucket=BUCKET,
        Key=key,
        Body=response.content,
        ContentType='image/jpeg'
    )

    # Return the public URL
    public_url = f"https://{BUCKET}.s3.{os.getenv('AWS_REGION')}.amazonaws.com/{key}"
    return public_url

url = upload_profile_pic("sarwatg", "https://instagram.fisb1-2.fna.fbcdn.net/v/t51.2885-19/363905047_813700833549836_8240770889961569719_n.jpg?efg=eyJ2ZW5jb2RlX3RhZyI6InByb2ZpbGVfcGljLmRqYW5nby44MDUuYzIifQ&_nc_ht=instagram.fisb1-2.fna.fbcdn.net&_nc_cat=108&_nc_oc=Q6cZ2gHVdVSb2KwYEHowYuwBGIrn9VgwAAmAFdBjGzc0vjjVv1gC609t9Fe13GWMhZDzVEA&_nc_ohc=GmMmG-p_swIQ7kNvwFyZXaM&_nc_gid=h_C_M_tjWNHwvfiVy1aRsw&edm=AP4sbd4BAAAA&ccb=7-5&oh=00_AfztYJxkB__fYAA_C_xbtAIv6qMOYSymjs1oJitP1TLteA&oe=69D0B9FE&_nc_sid=7a9f4b")
print(url)

https://upclout-profile-pics.s3.us-west-1.amazonaws.com/profile-pics/sarwatg.jpg


In [6]:
import boto3
import os
from dotenv import load_dotenv

load_dotenv()

s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
    region_name=os.getenv('AWS_REGION')
)

BUCKET = os.getenv('AWS_S3_BUCKET')
REGION = os.getenv('AWS_REGION')

def upload_all_profile_pics(base_path='../data/influencers_DONE'):
    uploaded = 0
    failed = []

    for username in os.listdir(base_path):
        folder = os.path.join(base_path, username)
        if not os.path.isdir(folder):
            continue

        jpg_path = os.path.join(folder, f"{username}.jpg")
        if not os.path.exists(jpg_path):
            failed.append((username, "No .jpg found"))
            continue

        try:
            key = f"profile-pics/{username}.jpg"
            s3.upload_file(
                jpg_path,
                BUCKET,
                key,
                ExtraArgs={'ContentType': 'image/jpeg'}
            )
            uploaded += 1
            print(f"✅ [{uploaded}] {username}")
        except Exception as e:
            failed.append((username, str(e)))
            print(f"❌ {username} — {e}")

    print(f"\nDone! Uploaded: {uploaded} | Failed: {len(failed)}")
    if failed:
        print("Failed uploads:")
        for name, reason in failed:
            print(f"  - {name}: {reason}")

    return uploaded, failed

In [8]:
upload_all_profile_pics()

✅ [1] a1nakhan_
✅ [2] aaleenk_
✅ [3] aamnaaqeel
✅ [4] aayrarahman
✅ [5] abbassbukharii
✅ [6] abdullah.ascends
✅ [7] acme_ana
✅ [8] adeenas.pov
✅ [9] adnslostfiles
✅ [10] aenaakhan
✅ [11] aesthetic_factor
✅ [12] afia_ahmd
✅ [13] afrahsblues
✅ [14] ahadrazamir
✅ [15] aimankhan.official
✅ [16] aimasibtain
✅ [17] aima_baig_official
✅ [18] aiza.___
✅ [19] aizara.j
✅ [20] alifsay
✅ [21] alirumman._
✅ [22] alishba.amirr
✅ [23] alishbabbasii
✅ [24] alishbahannjum
✅ [25] alizafatimaa98
✅ [26] alizehshahofficial
✅ [27] ali_zafar
✅ [28] alvibase
✅ [29] alymaliha
✅ [30] aminasultan__
✅ [31] amnakashifff
✅ [32] amnaniazi81
✅ [33] amnashpati
✅ [34] anas_shahid_official
✅ [35] annuralkhalid
✅ [36] anooshalala
✅ [37] areeba.__.asif
✅ [38] areeka__haq
✅ [39] areesharashidd
✅ [40] arslanaslamofficial
✅ [41] arya.saloka
✅ [42] ashikashaikh
✅ [43] asimazhar
✅ [44] asimjofa
✅ [45] asteriamusic_
✅ [46] ayesha.m.omar
✅ [47] ayeshajahangirmalik
✅ [48] ayeshakhalid_
✅ [49] ayezakhan.ak
✅ [50] aymen.saleem
✅ [5

(320,
 [('ayezakhanakhan', 'No .jpg found'),
  ('beanstheblogger', 'No .jpg found'),
  ('laib4.aaa', 'No .jpg found'),
  ('seshyaps', 'No .jpg found')])

In [5]:
import json

def extract_titles_to_profiles(json_path='saved_posts.json', profiles_path='../insta_profiles.txt'):
    """Extract all titles from saved_posts.json and append them to insta_profiles.txt (no duplicates with existing entries)."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    titles = [post['title'] for post in data.get('saved_saved_media', []) if 'title' in post]

    # Read existing profiles to avoid duplicates
    with open(profiles_path, 'r', encoding='utf-8') as f:
        existing = set(line.strip() for line in f if line.strip())

    # Filter out titles already present
    new_titles = [t for t in titles if t not in existing]
    # Also deduplicate within the new titles while preserving order
    seen = set()
    unique_new = []
    for t in new_titles:
        if t not in seen:
            seen.add(t)
            unique_new.append(t)

    # Append to file
    with open(profiles_path, 'a', encoding='utf-8') as f:
        for title in unique_new:
            f.write(title + '\n')

    print(f"Added {len(unique_new)} new profiles (skipped {len(titles) - len(unique_new)} duplicates)")
    return unique_new

# Run it
extract_titles_to_profiles()


Added 359 new profiles (skipped 81 duplicates)


['toobanaveedd',
 'manahils.drafts',
 'sanx_cosplay',
 'uswaatanveer',
 'alvibase',
 'youshmasdiary',
 'anooshalala',
 'hayyazaman',
 'n1mrah._x',
 'emannjafferr',
 'zohaazeem5_8',
 'minahillfatimah',
 'saraa.arj',
 'nehakarimullah',
 'mishatariq_01',
 'hassaniqbalrizvi',
 'm4ne4ter_',
 'annuralkhalid',
 'han1a.qurr',
 'tirzah.shams',
 'shopjewel._',
 'maheeen_ak',
 '_arizeh',
 'kiwiizara',
 'zeenatsulemann',
 'yumnaakamal',
 'uroojmumtazkhan',
 'zureenabnisar',
 'cat.boy.khai',
 'maryamfatimamalikk',
 'redhaireddesii',
 'roobz.liftz',
 'erfahhahmed',
 'mehro_.z',
 'marriyah.ali',
 'being.bayan',
 'ermdances',
 'nidarehmannn',
 'aayrarahman',
 'wrinkleinreality',
 'malaekaaa',
 'ayeshakhalid_',
 'librababbie',
 'beahhuseynn',
 'sanaakiyani',
 'rabea_mahmood_qureshi',
 'gymbyfatimah',
 'xmaheen._',
 'shredwithshanz',
 'acme_ana',
 'shanzeh.j',
 'i.ayeshu',
 'minalalala',
 'soniachaudhry_06',
 'khunshaaamir',
 '_nowrinkabir_',
 '_malkakhan',
 'thegirliegirl_',
 'emanmatloobb',
 'marrwwak

In [8]:
def remove_dup_influencers() -> None:

    with open("../insta_profiles.txt", "r") as file:
        influencers = [line.strip() for line in file.readlines()]

    size_with_duplicates: int = len(influencers)
    remove_duplicate_list = set(influencers)
    size_without_duplicates: int = len(remove_duplicate_list)

    if size_with_duplicates == size_without_duplicates:
        print("Already Up-to-date")
        return
    
    """with open("../insta_profiles.txt", "w") as file:
        for influencer in remove_duplicate_list:
            file.write(influencer + "\n")
    """
    print(f"Removed duplicates. {size_with_duplicates - size_without_duplicates} entries deleted.")


In [10]:
remove_dup_influencers()

Already Up-to-date


In [13]:
from load import Postgres
import psycopg2

In [14]:
conn = psycopg2.connect(database="postgres", user="postgres", password=1040)
cur = conn.cursor()

In [15]:
def does_mention_exist(username: str) -> bool:
    query = """
        SELECT p.caption
        FROM posts p
        JOIN influencers i
        ON p.ownerid = i.influencerid
        WHERE i.username = %s
    """
    cur = conn.cursor()
    cur.execute(query, (username,))
    result = cur.fetchall()
    cur.close()
    
    return result

In [16]:
res = does_mention_exist("mahirahkhan")
res[47]

('Little bit of Love Guru bts x \n\nP.S I don’t know how to jog and I begged my director to show me walking or something but he wanted a JOG and ugh it was tough. 🫣\n\nAlso notice how I had toot Gaya playing through every scene… whattta whattaa song! Can’t wait for it to release/ inshAllah.',)

In [29]:
def func():
    with open(r"C:\Users\ismai\Downloads\asmita.json", "r", encoding="utf-8") as file:
        data=json.load(file)

    return data

In [30]:
d = func()

In [34]:
if d[0]['private'] == True:
    print("skipping")

In [68]:
new_dict['mentions']

['mahirahkhan,',
 'rimpleandharpreet,',
 'hamnaamirjewelry',
 'sonia_ullah',
 'asadbinjavedphoto',
 'mues.concepts',
 'mannisahota_',
 'mrvikasrattu',
 'tanishqmalhotraa',
 'zahrasarfraz',
 'nupursarvaiya',
 'hairbyawais',
 'iambabarzaheer',
 'muzammilgarewal',
 'malaikapervezz',
 'thealizehdiaries',
 'fiza.mukaty']

In [5]:
mention_dict = {'PostID': 3704899387728828748,
 'mentions': ['mahirahkhan,',
  'rimpleandharpreet,',
  'hamnaamirjewelry',
  'sonia_ullah',
  'asadbinjavedphoto',
  'mues.concepts',
  'mannisahota_',
  'mrvikasrattu',
  'tanishqmalhotraa',
  'zahrasarfraz',
  'nupursarvaiya',
  'hairbyawais',
  'iambabarzaheer',
  'muzammilgarewal',
  'malaikapervezz',
  'thealizehdiaries',
  'fiza.mukaty']}

In [ ]:
dataframe = pd.DataFrame(mention_dict)
dataframe

,PostID,mentions
0,3704899387728828748,"mahirahkhan,"
1,3704899387728828748,"rimpleandharpreet,"
2,3704899387728828748,hamnaamirjewelry
3,3704899387728828748,sonia_ullah
4,3704899387728828748,asadbinjavedphoto
5,3704899387728828748,mues.concepts
6,3704899387728828748,mannisahota_
7,3704899387728828748,mrvikasrattu
8,3704899387728828748,tanishqmalhotraa
9,3704899387728828748,zahrasarfraz


In [164]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

url1="https://starngage.com/plus/en/brand/ranking/instagram/pakistan/politics"

url="https://starngage.com/plus/en/influencer/ranking/instagram/pakistan"
driver = webdriver.Chrome()  # or webdriver.Firefox()
driver.get(url)

# Wait for the table to load
wait = WebDriverWait(driver, 10)
tbody = wait.until(EC.presence_of_element_located((By.TAG_NAME, "tbody")))

# Find all name links
name_links = driver.find_elements(By.CSS_SELECTOR, "tbody tr .name a")
names = [link.text for link in name_links if link.text.strip()]

driver.quit()

cleaned_names = [name.lstrip('@') for name in names]

print(len(cleaned_names))

with open("../insta_profiles.txt", "a") as file:
    for username in cleaned_names:
        file.write(username + "\n")

100


In [140]:
with open("../data/brand_data.json", "r", encoding="utf-8") as file:
    data=json.load(file)

top_posts=data[1]['topPosts']

In [141]:
top_posts[29]['mentions']

[]

In [142]:
usernames=[]
mentions=[]

for index_2 in range(29):
    try:
        mentions.append(top_posts[index_2]['mentions'])
    except IndexError as e:
        print(f"Stopped at inner length: {index_2}\nError: {e}")

In [107]:
usernames=set(usernames)

In [143]:
mentions=[lst for lst in mentions if lst]
# Flatten the list
mentions = [username for sublist in mentions for username in sublist]

In [129]:
len(usernames)

0

In [144]:
len(mentions)

7

In [145]:
mentions

['nishatemporium',
 'Winter',
 'oriflamewithnazish',
 'am_brandstore',
 '03043888895',
 'khizan_official1',
 'khizanbts']

In [104]:
with open("../brand_profiles.txt", "a") as file:
    for username in usernames:
        file.write(username + "\n")
